<a href="https://colab.research.google.com/github/Midunvarsan/Codtech-Intern/blob/main/TEXT_SUMMARIZATION_TOOL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## `requirements.txt`

In [ ]:
%%writefile requirements.txt
torch
transformers
sentencepiece
gradio
scipy # Required by transformers
numpy # Required by transformers
pandas # For potential data handling within the tool if expanded, good to include for general ML projects

## `summarizer.py`

In [ ]:
%%writefile summarizer.py
import torch
from transformers import pipeline
import math

def load_model(model_name: str = "facebook/bart-large-cnn", device: int = -1):
    """
    Loads a pre-trained summarization model from Hugging Face Transformers.

    Args:
        model_name (str): The name of the pre-trained model to load.
                          Options: 'facebook/bart-large-cnn', 'google/pegasus-xsum', 't5-small'.
        device (int): Device to run the model on. -1 for CPU, 0 or greater for GPU id.

    Returns:
        transformers.pipelines.text2text_generation.SummarizationPipeline:
            A Hugging Face summarization pipeline object.

    Raises:
        ValueError: If an unsupported model_name is provided.
        RuntimeError: If there's an issue loading the model (e.g., connection error).
    """
    valid_models = ["facebook/bart-large-cnn", "google/pegasus-xsum", "t5-small"]
    if model_name not in valid_models:
        raise ValueError(f"Unsupported model: {model_name}. Choose from {valid_models}")

    try:
        print(f"Loading model: {model_name}...")
        # Use 'torch.cuda.is_available()' to check for GPU and assign device accordingly
        device_to_use = device if torch.cuda.is_available() else -1

        summarizer = pipeline(
            "summarization",
            model=model_name,
            tokenizer=model_name,
            device=device_to_use # Assign device here
        )
        print(f"Model {model_name} loaded successfully on device {device_to_use}.")
        return summarizer
    except Exception as e:
        raise RuntimeError(f"Error loading model {model_name}: {e}")

def chunk_text(text: str, max_chunk_length: int = 1024, overlap: int = 100) -> list[str]:
    """
    Splits a long text into smaller chunks, handling token limits for Transformer models.

    Args:
        text (str): The input text to be chunked.
        max_chunk_length (int): The maximum token length for each chunk (model's input limit).
        overlap (int): The number of tokens to overlap between consecutive chunks to maintain context.

    Returns:
        list[str]: A list of text chunks.
    """
    if not text.strip():
        return []

    words = text.split()
    # Estimate token count by word count for simplicity. For exact, need tokenizer.
    # Transformers tokenizers typically count sub-word units, so word count is a rough upper bound.
    approx_tokens_per_word = 1.3 # A common estimate for English text
    max_words_per_chunk = math.floor(max_chunk_length / approx_tokens_per_word)
    overlap_words = math.floor(overlap / approx_tokens_per_word)

    chunks = []
    current_start = 0
    while current_start < len(words):
        current_end = current_start + max_words_per_chunk
        chunk = " ".join(words[current_start:current_end])
        chunks.append(chunk)
        current_start += (max_words_per_chunk - overlap_words)
        if current_start >= len(words): # Ensure we don't go out of bounds on the last chunk logic
            break

    return chunks

def generate_summary(summarizer_pipeline, text: str, min_length_ratio: float = 0.1, max_length_ratio: float = 0.3) -> str:
    """
    Generates an abstractive summary of the input text using the provided summarizer pipeline.
    Handles chunking for texts exceeding the model's token limit.

    Args:
        summarizer_pipeline: The Hugging Face summarization pipeline.
        text (str): The input text to be summarized.
        min_length_ratio (float): Desired minimum summary length as a ratio of original text words.
        max_length_ratio (float): Desired maximum summary length as a ratio of original text words.

    Returns:
        str: The generated summary.

    Raises:
        ValueError: If input text is empty or invalid ratios are provided.
        RuntimeError: If summarization fails.
    """
    if not text or not text.strip():
        raise ValueError("Input text cannot be empty for summarization.")
    if not (0 < min_length_ratio < 1) or not (0 < max_length_ratio < 1) or (min_length_ratio >= max_length_ratio):
        raise ValueError("Invalid summary length ratios. min_length_ratio must be less than max_length_ratio, and both must be between 0 and 1.")

    original_word_count = len(text.split())

    # Determine actual min_length and max_length for the model
    # The 'max_length' parameter in summarization pipelines typically refers to output tokens.
    # A rough estimate is 1 token per 1.5 words for output.
    min_output_words = max(10, int(original_word_count * min_length_ratio))
    max_output_words = max(20, int(original_word_count * max_length_ratio))

    min_length_tokens = math.ceil(min_output_words * 1.5)
    max_length_tokens = math.ceil(max_output_words * 1.5)

    # Ensure min_length is not greater than max_length for the model
    if min_length_tokens >= max_length_tokens:
        min_length_tokens = max(10, max_length_tokens // 2) # Fallback to a reasonable min

    try:
        # Check model's max input length
        # Some models define this in config. If not, a common default is 1024.
        max_input_length = summarizer_pipeline.model.config.max_position_embeddings
        # BART-large-cnn typically has 1024 input tokens.
        # If model config doesn't expose it or it's too large (e.g., Longformer), cap it.
        if max_input_length > 1024 or max_input_length is None:
            max_input_length = 1024

        chunks = chunk_text(text, max_chunk_length=max_input_length, overlap=100)

        if not chunks:
            return ""

        # Summarize each chunk if text is long, then combine or re-summarize
        # For simplicity, if multiple chunks, we summarize each and concatenate.
        # A more advanced approach might involve hierarchical summarization.
        chunk_summaries = []
        for i, chunk in enumerate(chunks):
            print(f"Summarizing chunk {i+1}/{len(chunks)}...")
            # Adjust min/max length for chunks if it's not the final pass
            chunk_min_length = max(5, min_length_tokens // len(chunks) if len(chunks) > 1 else min_length_tokens)
            chunk_max_length = max(10, max_length_tokens // len(chunks) if len(chunks) > 1 else max_length_tokens)

            # Ensure chunk_min_length <= chunk_max_length
            if chunk_min_length > chunk_max_length:
                chunk_min_length = chunk_max_length // 2 # Ensure it's not larger

            summary_output = summarizer_pipeline(
                chunk,
                min_length=chunk_min_length,
                max_length=chunk_max_length,
                do_sample=False # For more deterministic output
            )
            chunk_summaries.append(summary_output[0]['summary_text'])

        final_summary = " ".join(chunk_summaries).strip()

        # Optional: if the combined summary is still too long, re-summarize it
        # This is a simple approach. For production, consider multi-stage summarization.
        if len(final_summary.split()) > max_output_words * 1.5 and len(chunks) > 1:
            print("Re-summarizing combined chunks...")
            re_summary_output = summarizer_pipeline(
                final_summary,
                min_length=min_length_tokens, # Use original target lengths
                max_length=max_length_tokens,
                do_sample=False
            )
            final_summary = re_summary_output[0]['summary_text']

        return final_summary
    except Exception as e:
        raise RuntimeError(f"Error during summary generation: {e}")

## `app.py`

In [ ]:
%%writefile app.py
import gradio as gr
import time
from summarizer import load_model, generate_summary

# --- Global Model and Pipeline Initialization (Cached) ---
# This will load the model only once when the app starts
# You can choose the default model here. bart-large-cnn is generally good.
# device=-1 for CPU, 0 for GPU (if available)

# Try to load BART, if not available, fall back to Pegasus, then t5-small
try:
    print("Attempting to load facebook/bart-large-cnn...")
    summarizer_pipeline = load_model("facebook/bart-large-cnn")
except RuntimeError as e:
    print(f"Could not load facebook/bart-large-cnn: {e}")
    try:
        print("Attempting to load google/pegasus-xsum as a fallback...")
        summarizer_pipeline = load_model("google/pegasus-xsum")
    except RuntimeError as e:
        print(f"Could not load google/pegasus-xsum: {e}")
        print("Loading t5-small as a lightweight fallback...")
        summarizer_pipeline = load_model("t5-small") # Final fallback

# --- Helper function for Gradio Interface ---
def summarize_text_ui(text: str, summary_length: str) -> tuple[str, str, str, str]:
    """
    Gradio-compatible function to summarize text and provide analytics.

    Args:
        text (str): The input text from the user.
        summary_length (str): Desired summary length ('Short', 'Medium', 'Long').

    Returns:
        tuple[str, str, str, str]: (summary, original_word_count_str, summarized_word_count_str, compression_ratio_str)
    """
    if not text or not text.strip():
        return "Please enter some text to summarize.", "N/A", "N/A", "N/A"

    original_words = text.split()
    original_word_count = len(original_words)

    # Map summary length choice to min/max ratios
    if summary_length == "Short":
        min_ratio, max_ratio = 0.05, 0.15
    elif summary_length == "Medium":
        min_ratio, max_ratio = 0.15, 0.30
    else: # Long
        min_ratio, max_ratio = 0.30, 0.50

    start_time = time.time()
    try:
        summary = generate_summary(summarizer_pipeline, text, min_length_ratio=min_ratio, max_length_ratio=max_ratio)
        end_time = time.time()
        print(f"Summarization took {end_time - start_time:.2f} seconds.")

        summarized_words = summary.split()
        summarized_word_count = len(summarized_words)

        if original_word_count > 0:
            compression_ratio = (1 - (summarized_word_count / original_word_count)) * 100
        else:
            compression_ratio = 0.0

        return (
            summary,
            f"{original_word_count} words",
            f"{summarized_word_count} words",
            f"{compression_ratio:.2f}%"
        )
    except Exception as e:
        return f"An error occurred during summarization: {e}", "N/A", "N/A", "N/A"

# --- Gradio Interface Definition ---
with gr.Blocks(theme=gr.themes.Soft(primary_hue=gr.Color("#4CAF50"))) as demo:
    gr.Markdown(
        """
        # 📝 Production-Grade Text Summarization Tool
        """
    )

    with gr.Row():
        with gr.Column():
            text_input = gr.Textbox(
                label="Paste your article here",
                lines=20,
                placeholder="Enter a long article or document to summarize..."
            )
            summary_length_dropdown = gr.Dropdown(
                ["Short", "Medium", "Long"],
                label="Desired Summary Length",
                value="Medium"
            )
            summarize_button = gr.Button("Summarize Article")

        with gr.Column():
            summary_output = gr.Textbox(
                label="Generated Summary",
                lines=15,
                interactive=False # Users cannot type here
            )
            with gr.Column():
                gr.Markdown("### Analytics")
                original_wc = gr.Textbox(label="Original Word Count", interactive=False)
                summarized_wc = gr.Textbox(label="Summarized Word Count", interactive=False)
                compression_ratio = gr.Textbox(label="Compression Ratio", interactive=False)

    summarize_button.click(
        fn=summarize_text_ui,
        inputs=[text_input, summary_length_dropdown],
        outputs=[summary_output, original_wc, summarized_wc, compression_ratio]
    )

    gr.Examples(
        examples=[
            ["""
            The Amazon rainforest is the largest tropical rainforest in the world, covering an immense area across nine South American countries: Brazil, Peru, Ecuador, Colombia, Venezuela, Guyana, Suriname, Bolivia, and French Guiana. Roughly 60% of the rainforest is in Brazil. It is home to an incredible array of biodiversity, housing an estimated 10% of the world's known species, including countless insects, birds, mammals, and plant life. The Amazon River, which flows through the forest, is the second-longest river in the world by length and the largest by discharge volume, carrying more water than the next seven largest rivers combined. This vast ecosystem plays a crucial role in regulating the Earth's climate by absorbing vast amounts of carbon dioxide and producing oxygen, earning it the nickname 'the Lungs of the Earth.' However, the Amazon is under constant threat from deforestation due to agriculture, logging, mining, and infrastructure development, leading to significant environmental concerns and loss of habitat for its unique wildlife.
            """, "Medium"],
            ["""
            Artificial intelligence (AI) is rapidly transforming various aspects of human society, from healthcare to finance, and transportation to entertainment. Machine learning, a subset of AI, has enabled systems to learn from data without explicit programming, leading to breakthroughs in areas like image recognition, natural language processing, and predictive analytics. Deep learning, a further specialization within machine learning, utilizes neural networks with multiple layers to uncover intricate patterns in vast datasets, powering applications such as autonomous vehicles and sophisticated recommendation systems. The ethical implications of AI, including bias in algorithms, job displacement, and privacy concerns, are subjects of ongoing debate and research. As AI continues to evolve, its potential to address complex global challenges, such as climate change and disease, is immense, but responsible development and deployment are paramount to ensure its benefits are broadly shared and its risks are mitigated.
            """, "Medium"]
        ],
        inputs=[text_input, summary_length_dropdown]
    )

# To launch the Gradio interface
# demo.launch(share=True) # Use share=True to get a public link
# When running in Colab, this will usually provide a public URL automatically.

if __name__ == "__main__":
    demo.launch(debug=True, share=True)